# HCP Position + Hierarchical Cone Direction + Energy Optimization
## Advanced three-stage optimization: HCP lattice position search + hierarchical cone direction search + energy optimization

This notebook implements a state-of-the-art optimization approach:
- **Stage 1**: HCP lattice grid search for optimal origin position using origin_time_loss
- **Stage 2**: Hierarchical cone-based direction search with adaptive refinement
- **Stage 3**: Energy scan around initial guess E_guess = 1.782 * N^0.674 + -97.460
- **Stage 4**: Combined loss gradient optimization (position + direction + t0 + energy)
- **Enhanced 3D Visualization**: Complete optimization trajectory with cone search visualization

**Key Innovation**: No longer assumes known energy - estimates energy from photon count then optimizes energy alongside other parameters.

In [ ]:
import sys
sys.path.append('..')

from tools.geometry import generate_detector
from tools.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from tools.losses import WC_loss
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim

import jax
import jax.numpy as jnp
import time
import math

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from scipy.interpolate import interp1d
import optax
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize

In [ ]:
# Configuration
default_json_filename = '../config/SK_geom_config.json'
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
TEMPERATURE = 0.1  # Fixed temperature
N_EVENTS = 30  # Number of events for analysis
K = 5
Nphot = 1_000_000

# Product loss optimization parameters
VERTEX_WEIGHT_SCALE = 1.0    # Scaling factor for vertex loss gradient contribution
WC_WEIGHT_SCALE = 1.0        # Scaling factor for WC loss gradient contribution
ENERGY_WEIGHT_SCALE = 1.0    # Scaling factor for energy loss gradient contribution
POSITION_LEARNING_RATE = 0.5 # Learning rate for position updates
DIRECTION_LEARNING_RATE = 0.1 # Learning rate for direction updates
T0_LEARNING_RATE = 0.5       # Learning rate for t0 updates
ENERGY_LEARNING_RATE = 10000.   # Learning rate for energy updates

# HCP lattice position grid search parameters
HCP_L0 = 6.0                 # Initial lattice spacing for HCP grid
HCP_LEVELS = 4               # Number of HCP refinement levels
HCP_REDUCTION = 0.5          # Reduction factor between HCP levels

# Hierarchical cone direction search parameters
CONE_LEVELS = 2              # Number of hierarchical levels (as requested)
CONE_INITIAL_DIV = 5         # Initial divisions for global sphere sampling
CONE_MAX_ANGLE_DEG = 30      # Maximum cone opening angle in degrees
CONE_REDUCTION = 0.25        # Cone angle reduction factor between levels

# Energy estimation and scan parameters
ENERGY_DELTA = 200           # Energy scan range (±E_delta around E_guess)
ENERGY_SCAN_STEPS = 25       # Number of energy scan steps

# Gradient descent iterations
MAX_ITERATIONS = 1000

# Visualization parameters
ARROW_EVERY_N_POINTS = 100  # Show direction arrow every N optimization points

# Speed of light in medium
C_MEDIUM = 1.0

# Setup detector
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)
DETECTOR_R = detector.r
DETECTOR_H = detector.H

# Setup prediction simulator with fixed temperature (is_data=False)
prediction_simulator = setup_event_simulator(default_json_filename, Nphot, TEMPERATURE, K=K, is_data=False)

# Setup data simulator for generating target events (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=K,
                                      is_data=True, is_calibration=False)

print(f"Number of sensors: {NUM_DETECTORS}")
print(f"Detector dimensions: R={DETECTOR_R:.1f}m, H={DETECTOR_H:.1f}m")
print(f"Fixed temperature: {TEMPERATURE}")
print(f"Speed of light in medium: {C_MEDIUM}")
print(f"Number of events for analysis: {N_EVENTS}")
print(f"HCP lattice parameters:")
print(f"  Initial spacing L0: {HCP_L0}")
print(f"  Refinement levels: {HCP_LEVELS}")
print(f"  Reduction factor: {HCP_REDUCTION}")
print(f"Hierarchical cone direction search parameters:")
print(f"  Levels: {CONE_LEVELS}")
print(f"  Initial divisions: {CONE_INITIAL_DIV}")
print(f"  Max cone angle: {CONE_MAX_ANGLE_DEG}°")
print(f"  Cone reduction factor: {CONE_REDUCTION}")
print(f"Energy optimization parameters:")
print(f"  Energy delta: ±{ENERGY_DELTA}")
print(f"  Energy scan steps: {ENERGY_SCAN_STEPS}")

# Load ROOT file information
with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

# Detector parameters
detector_params = (
    jnp.array(50.),           # scatter_length
    jnp.array(0.2),          # reflection_rate
    jnp.array(50.),           # absorption_length
    jnp.array(0.001)         # gumbel_softmax_temp
)

In [ ]:
# Energy estimation function
@jit
def estimate_energy_from_photon_count(N):
    """
    Estimate energy from photon count using empirical fit:
    E_guess = 1.782 * N^0.674 + -97.460
    
    Parameters:
    -----------
    N : int or jnp.ndarray
        Number of observed photons
        
    Returns:
    --------
    float
        Estimated energy
    """
    return 1.782 * jnp.power(N, 0.674) - 97.460

@jit
def energy_loss(simulated_charge, true_charge):
    """
    Core energy loss computation using intensity matching.
    
    Parameters:
    -----------
    simulated_charge : jnp.ndarray
        Simulated charge values
    true_charge : jnp.ndarray
        True charge values
        
    Returns:
    --------
    float
        Energy loss value
    """
    total_true_charge = jnp.sum(true_charge)
    total_sim_charge = jnp.sum(simulated_charge)
    eps = 1e-8

    return jnp.abs(jnp.log(total_sim_charge / (total_true_charge + eps)))

print("Energy estimation and loss functions defined")

In [ ]:
# HCP lattice grid generation functions
def cylinder_hcp_points_local(center_xy, z_center, R_local, H_local, L, R, H):
    """
    Generate HCP lattice points inside a local cylinder.
    - center_xy, z_center: center of local cylinder
    - R_local, H_local: half-sizes of local cylinder
    - L: nearest-neighbor spacing
    - R, H: global cylinder dimensions
    """
    cx, cy = center_xy
    cz = z_center

    # in-plane triangular lattice spacing
    dx = L
    dy = math.sqrt(3) / 2 * L
    dz = math.sqrt(3) / 2 * L  # vertical spacing

    # bounds of local search region
    x_min, x_max = cx - R_local, cx + R_local
    y_min, y_max = cy - R_local, cy + R_local
    z_min, z_max = cz - H_local, cz + H_local

    xs = np.arange(x_min, x_max + dx, dx)
    ys = np.arange(y_min, y_max + dy, dy)
    zs = np.arange(z_min, z_max + dz, dz)

    pts = []
    for i, z in enumerate(zs):
        z_global = z
        # if z_global < 0 or z_global > H:
        #     continue

        layer_shift_x = 0.0
        layer_shift_y = 0.0
        if i % 2 == 1:
            layer_shift_x = 0.5 * dx
            layer_shift_y = 0.5 * dy

        for ix, x in enumerate(xs):
            for iy, y in enumerate(ys):
                xg = x + (iy % 2) * 0.5 * dx + layer_shift_x
                yg = y + layer_shift_y

                # check global cylinder constraint
                if (xg**2 + yg**2) <= R**2:
                    pts.append((xg, yg, z_global))

    return np.array(pts)

In [ ]:
# Hierarchical cone-based direction search functions
def cone_points(axis, max_angle_rad, num_rings):
    """
    Generate points on a cone around 'axis' (unit vector).
    max_angle_rad: maximum opening angle of the cone
    num_rings: number of rings along the opening angle
    Returns array of points (N x 3)
    """
    axis = axis / np.linalg.norm(axis)

    # Find two orthogonal vectors perpendicular to axis
    if abs(axis[2]) < 0.99:
        u = np.cross(axis, [0,0,1])
    else:
        u = np.cross(axis, [0,1,0])
    u /= np.linalg.norm(u)
    v = np.cross(axis, u)

    points = []

    for i in range(num_rings):
        theta = (i+1) * max_angle_rad / num_rings  # opening angle from axis
        ring_radius = np.sin(theta)
        z = np.cos(theta)

        # number of points on ring to match spacing along theta
        num_points_ring = max(1, int(2*np.pi*ring_radius*num_rings))
        for j in range(num_points_ring):
            phi = 2*np.pi*j / num_points_ring
            point = z*axis + ring_radius*(np.cos(phi)*u + np.sin(phi)*v)
            points.append(point)

    return np.array(points)


def hierarchical_direction_search_cone(position, initial_t0, hit_detector_positions, 
                                     observed_times, observed_charge, true_data, energy_guess,
                                     levels=CONE_LEVELS, initial_div=CONE_INITIAL_DIV, 
                                     max_angle_deg=CONE_MAX_ANGLE_DEG, reduction=CONE_REDUCTION):
    """
    Hierarchical cone-based direction search using combined loss evaluation.
    
    Args:
        position: [x, y, z] position (from HCP search)
        initial_t0: starting t0 value
        energy_guess: energy estimate from photon count
        levels: number of hierarchical levels
        initial_div: initial divisions for global sphere sampling
        max_angle_deg: maximum cone opening angle in degrees
        reduction: cone angle reduction factor between levels
    
    Returns:
        dict with optimal direction and hierarchical search results
    """
    
    print(f"    Performing {levels}-level hierarchical cone direction search...")
    print(f"    Parameters: initial_div={initial_div}, max_angle={max_angle_deg}°, reduction={reduction}")
    
    # Initial global grid over sphere
    num_theta, num_phi = initial_div, initial_div*2
    thetas = np.linspace(0, np.pi, num_theta)
    phis = np.linspace(0, 2*np.pi, num_phi, endpoint=False)
    directions = np.array([[np.sin(t)*np.cos(p), np.sin(t)*np.sin(p), np.cos(t)] 
                          for t in thetas for p in phis])
    
    best_direction = None
    max_angle_rad = np.radians(max_angle_deg)
    path = []
    
    # Random key for loss evaluations
    search_key = jax.random.PRNGKey(789)
    
    for lvl in range(levels):
        print(f"      Level {lvl}: Evaluating {len(directions)} directions")
        
        # Evaluate combined loss for each direction
        level_results = []
        best_loss = float('inf')
        best_level_direction = None
        
        for i, direction in enumerate(directions):
            # Convert direction to spherical coordinates
            theta = np.arccos(np.clip(direction[2], -1.0, 1.0))
            phi = np.arctan2(direction[1], direction[0])
            
            # Create parameter vector for this direction
            test_params = jnp.array([
                position[0], position[1], position[2],
                initial_t0, theta, phi, energy_guess
            ])
            
            search_key, _ = jax.random.split(search_key)
            
            try:
                # Evaluate combined loss at this direction
                combined_loss, vertex_loss, wc_loss, energy_loss_val = combined_product_loss_with_energy(
                    test_params, hit_detector_positions, observed_times, observed_charge,
                    true_data, detector_params, search_key
                )
                
                direction_result = {
                    'direction': direction.copy(),
                    'theta': float(theta),
                    'phi': float(phi),
                    'combined_loss': float(combined_loss),
                    'vertex_loss': float(vertex_loss),
                    'wc_loss': float(wc_loss),
                    'energy_loss': float(energy_loss_val)
                }
                
                level_results.append(direction_result)
                
                # Track best result for this level
                if combined_loss < best_loss:
                    best_loss = combined_loss
                    best_level_direction = direction.copy()
                    
            except Exception as e:
                print(f"        Error evaluating direction {i}: {e}")
                continue
        
        if best_level_direction is None:
            print(f"      ERROR: No valid directions found at level {lvl}")
            break
            
        # Store level results
        level_summary = {
            'level': lvl,
            'num_directions': len(directions),
            'directions': directions.copy(),
            'direction_results': level_results,
            'best_direction': best_level_direction.copy(),
            'best_loss': best_loss,
            'max_angle_rad': max_angle_rad
        }
        
        path.append(level_summary)
        best_direction = best_level_direction
        
        print(f"      Level {lvl} best loss: {best_loss:.6f}")
        print(f"      Level {lvl} best direction: {best_level_direction}")
        
        # Prepare next level: generate cone around best direction
        if lvl < levels - 1:  # Don't generate cone for last level
            num_rings = 4  # Fixed number of rings for cone
            directions = cone_points(best_level_direction, max_angle_rad, num_rings)
            
            # Shrink cone angle for next level
            max_angle_rad *= reduction
    
    # Convert final best direction to spherical coordinates
    final_theta = np.arccos(np.clip(best_direction[2], -1.0, 1.0))
    final_phi = np.arctan2(best_direction[1], best_direction[0])
    
    print(f"    Hierarchical cone search complete. Best loss: {path[-1]['best_loss']:.6f}")
    print(f"    Best direction: {best_direction}")
    print(f"    Best angles: θ={final_theta:.3f}, φ={final_phi:.3f}")
    
    return {
        'best_direction': best_direction,
        'best_theta': float(final_theta),
        'best_phi': float(final_phi),
        'best_loss': path[-1]['best_loss'],
        'search_path': path,
        'total_levels': len(path)
    }

print("Hierarchical cone direction search functions defined")

In [ ]:
# Combined loss functions with energy
@jit
def origin_time_loss(origin, detector_positions, true_times, true_q, t0):
    """Vertex time loss component"""
    distances = jnp.linalg.norm(detector_positions - origin[None, :], axis=1)
    expected_times = (distances - 0.25/2) / C_MEDIUM
    time_residuals = true_times - expected_times - t0
    
    threshold = jnp.percentile(true_q, 85)
    w = jnp.where(true_q > threshold, true_q, 0.)
    
    neg_time_res = jnp.where(time_residuals < 0, jnp.abs(time_residuals), 0.0)
    pos_time_res = jnp.where(time_residuals > 0, jnp.abs(time_residuals), 0.0)
    
    neg_loss_per_detector = neg_time_res * w 
    pos_loss_per_detector = pos_time_res * w 
    
    total_loss = jnp.sum(neg_loss_per_detector) / (jnp.sum(w) + 1e-8) + jnp.sum(jnp.abs(pos_loss_per_detector * w)) / (jnp.sum(w) + 1e-8) / 1000.
    
    return total_loss

@jit
def wc_loss_component(position, theta, phi, energy, true_data, detector_params, key):
    """WC loss component"""
    spherical_params = (energy, position, jnp.array([theta, phi]))
    
    simulated_data = prediction_simulator(spherical_params, detector_params, key)
    
    loss = WC_loss(
        detector_points, *true_data, *simulated_data,
        lambda_poisson=1.0,
        lambda_time=1.0
    )
    
    return loss, simulated_data

@jit
def energy_loss_component(position, theta, phi, energy, true_data, detector_params, key, observed_charge):
    """Energy loss component using intensity matching"""
    spherical_params = (energy, position, jnp.array([theta, phi]))
    
    simulated_data = prediction_simulator(spherical_params, detector_params, key)
    simulated_charge = simulated_data[0]  # First element is charge
    
    return energy_loss(simulated_charge, observed_charge)

@jit
def combined_product_loss_with_energy(params, hit_detector_positions, observed_times, observed_charge, 
                                    true_data, detector_params, key, 
                                    vertex_weight=VERTEX_WEIGHT_SCALE, wc_weight=WC_WEIGHT_SCALE, 
                                    energy_weight=ENERGY_WEIGHT_SCALE):
    """
    Combined loss function: product of vertex loss, WC loss, and energy loss
    
    Args:
        params: [x, y, z, t0, theta, phi, energy] where theta and phi are spherical direction angles
        vertex_weight: scaling factor for vertex loss contribution
        wc_weight: scaling factor for WC loss contribution  
        energy_weight: scaling factor for energy loss contribution
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]
    
    # Calculate individual loss components
    vertex_loss = origin_time_loss(position, hit_detector_positions, observed_times, observed_charge, t0)
    wc_loss, simulated_data = wc_loss_component(position, theta, phi, energy, true_data, detector_params, key)
    energy_loss_val = energy_loss(simulated_data[0], observed_charge)  # Use simulated charge from WC component
    
    # Weighted product loss with small offset to avoid zero
    combined_loss = jnp.sqrt((vertex_weight * vertex_loss + 1e-6) * (wc_weight * wc_loss + 1e-6))# * (energy_weight * energy_loss_val + 1e-6))
    
    return combined_loss, vertex_loss, wc_loss, energy_loss_val

@jit
def spherical_to_cartesian(theta, phi):
    """Convert spherical angles to Cartesian direction vector"""
    sin_theta = jnp.sin(theta)
    cos_theta = jnp.cos(theta)
    sin_phi = jnp.sin(phi)
    cos_phi = jnp.cos(phi)
    
    return jnp.array([sin_theta * cos_phi, sin_theta * sin_phi, cos_theta])

@jit
def cartesian_to_spherical(direction):
    """Convert Cartesian direction vector to spherical angles"""
    # Normalize direction
    direction = direction / (jnp.linalg.norm(direction) + 1e-8)
    
    theta = jnp.arccos(jnp.clip(direction[2], -1.0, 1.0))
    phi = jnp.arctan2(direction[1], direction[0])
    
    return theta, phi

# Create gradient function for the combined loss with energy
combined_grad_fn_with_energy = jit(value_and_grad(lambda *args: combined_product_loss_with_energy(*args)[0]))

print("Combined product loss functions with energy defined")

In [ ]:
def hcp_position_grid_search(hit_detector_positions, observed_times, observed_charge, 
                            true_position, true_t0, R=DETECTOR_R, H=DETECTOR_H,
                            L0=HCP_L0, levels=HCP_LEVELS, reduction=HCP_REDUCTION):
    """
    Perform HCP lattice grid search for optimal origin position using origin_time_loss
    
    Args:
        hit_detector_positions: positions of detectors with hits
        observed_times: timing data
        observed_charge: charge data 
        true_position: true position for comparison
        true_t0: true t0 for loss evaluation
        R, H: detector cylinder dimensions
        L0: initial HCP lattice spacing
        levels: number of refinement levels
        reduction: reduction factor between levels
    
    Returns:
        dict with HCP search results and optimal position
    """
    
    print(f"    Performing HCP lattice position grid search...")
    print(f"    Parameters: L0={L0}, levels={levels}, reduction={reduction}")
    print(f"    Detector dimensions: R={R:.1f}m, H={H:.1f}m")
    
    # Generate all HCP levels
    all_hcp_results = []
    best_overall_loss = float('inf')
    best_overall_position = None
    
    # Start from detector center
    center_xy = (0.0, 0.0)
    z_center = 0.0
    R_local = R
    H_local = H / 2.0
    L = L0
    
    for level in range(levels):
        print(f"      HCP Level {level}: L={L:.3f}, R_local={R_local:.3f}, H_local={H_local:.3f}")
        
        # Generate HCP lattice points for this level
        hcp_points = cylinder_hcp_points_local(center_xy, z_center, R_local, H_local, L, R, H)
        
        if len(hcp_points) == 0:
            print(f"      No valid HCP points at level {level}")
            break
            
        print(f"      Generated {len(hcp_points)} HCP points")
        
        # Evaluate origin_time_loss at each HCP point
        level_results = []
        best_level_loss = float('inf')
        best_level_position = None
        
        for i, point in enumerate(hcp_points):
            position = jnp.array(point)
            
            try:
                # Evaluate origin_time_loss
                loss = origin_time_loss(position, hit_detector_positions, 
                                      observed_times, observed_charge, true_t0)
                
                level_results.append({
                    'position': np.array(position),
                    'loss': float(loss),
                    'distance_to_true': float(jnp.linalg.norm(position - true_position))
                })
                
                # Track best for this level
                if loss < best_level_loss:
                    best_level_loss = loss
                    best_level_position = position
                    
                # Track best overall
                if loss < best_overall_loss:
                    best_overall_loss = loss
                    best_overall_position = position
                    
            except Exception as e:
                print(f"        Error evaluating point {i}: {e}")
                continue
        
        level_summary = {
            'level': level,
            'L': L,
            'center_xy': center_xy,
            'z_center': z_center,
            'R_local': R_local,
            'H_local': H_local,
            'num_points': len(hcp_points),
            'hcp_points': hcp_points,
            'point_results': level_results,
            'best_position': np.array(best_level_position) if best_level_position is not None else None,
            'best_loss': best_level_loss
        }
        
        all_hcp_results.append(level_summary)
        
        print(f"      Level {level} best loss: {best_level_loss:.6f}")
        print(f"      Level {level} best position: {best_level_position}")
        
        # Prepare for next level refinement
        if best_level_position is not None:
            center_xy = (float(best_level_position[0]), float(best_level_position[1]))
            z_center = float(best_level_position[2])
            
            # Shrink search region for next level
            L_next = L * reduction
            R_local = math.sqrt(3) / 2 * L/2
            H_local = math.sqrt(3) / 2 * L/2
            L = L_next
        else:
            break
    
    # Calculate final statistics
    final_position_error = float(jnp.linalg.norm(best_overall_position - true_position)) if best_overall_position is not None else float('inf')
    
    print(f"    HCP search complete. Best overall loss: {best_overall_loss:.6f}")
    print(f"    Best position: {best_overall_position}")
    print(f"    Position error: {final_position_error:.3f}m")
    
    return {
        'hcp_levels': all_hcp_results,
        'best_position': np.array(best_overall_position) if best_overall_position is not None else None,
        'best_loss': best_overall_loss,
        'position_error': final_position_error,
        'total_levels': len(all_hcp_results)
    }

print("HCP position grid search function defined")

In [ ]:
def energy_scan_optimization(position, theta, phi, initial_t0, hit_detector_positions,
                           observed_times, observed_charge, true_data, energy_guess,
                           energy_delta=ENERGY_DELTA, n_steps=ENERGY_SCAN_STEPS):
    """
    Perform energy scan around initial energy guess to find optimal energy.
    
    Args:
        position: [x, y, z] position from HCP search
        theta, phi: direction angles from cone search
        initial_t0: t0 estimate
        energy_guess: initial energy estimate from photon count
        energy_delta: scan range (±energy_delta)
        n_steps: number of scan steps
    
    Returns:
        dict with energy scan results and optimal energy
    """
    
    print(f"    Performing energy scan around E_guess={energy_guess:.1f}")
    print(f"    Scan range: [{energy_guess-energy_delta:.1f}, {energy_guess+energy_delta:.1f}] in {n_steps} steps")
    
    # Generate energy scan points
    energy_min = energy_guess - energy_delta
    energy_max = energy_guess + energy_delta
    energy_scan_points = np.linspace(energy_min, energy_max, n_steps)
    
    scan_results = []
    best_energy = energy_guess
    best_loss = float('inf')
    
    # Random key for loss evaluations
    scan_key = jax.random.PRNGKey(456)
    
    for i, energy in enumerate(energy_scan_points):
        # Create parameter vector for this energy
        test_params = jnp.array([
            position[0], position[1], position[2],
            initial_t0, theta, phi, energy
        ])
        
        scan_key, _ = jax.random.split(scan_key)
        
        try:
            # Evaluate combined loss at this energy
            combined_loss, vertex_loss, wc_loss, energy_loss_val = combined_product_loss_with_energy(
                test_params, hit_detector_positions, observed_times, observed_charge,
                true_data, detector_params, scan_key
            )
            
            energy_result = {
                'energy': float(energy),
                'combined_loss': float(combined_loss),
                'vertex_loss': float(vertex_loss),
                'wc_loss': float(wc_loss),
                'energy_loss': float(energy_loss_val)
            }
            
            scan_results.append(energy_result)
            
            # Track best energy
            if combined_loss < best_loss:
                best_loss = combined_loss
                best_energy = energy
                
        except Exception as e:
            print(f"        Error evaluating energy {energy:.1f}: {e}")
            continue
    
    print(f"    Energy scan complete. Best energy: {best_energy:.1f} (loss: {best_loss:.6f})")
    
    return {
        'energy_guess': float(energy_guess),
        'energy_min': float(energy_min),
        'energy_max': float(energy_max),
        'n_steps': n_steps,
        'scan_results': scan_results,
        'best_energy': float(best_energy),
        'best_loss': best_loss,
        'energy_improvement': float(abs(best_energy - energy_guess))
    }

print("Energy scan optimization function defined")

In [ ]:
def run_complete_hcp_cone_energy_optimization(initial_t0, hit_detector_positions, observed_times, observed_charge,
                                            true_data, true_energy, true_position, true_direction, TRUE_T0,
                                            pos_lr=POSITION_LEARNING_RATE, dir_lr=DIRECTION_LEARNING_RATE, 
                                            t0_lr=T0_LEARNING_RATE, energy_lr=ENERGY_LEARNING_RATE,
                                            vertex_weight=VERTEX_WEIGHT_SCALE, wc_weight=WC_WEIGHT_SCALE, 
                                            energy_weight=ENERGY_WEIGHT_SCALE,
                                            max_iterations=3000, tolerance=1e-6):
    """
    Run complete optimization pipeline with energy:
    1. Energy estimation from photon count
    2. HCP lattice position grid search
    3. Hierarchical cone direction search
    4. Energy scan optimization
    5. Combined gradient optimization (position + direction + t0 + energy)
    
    Args:
        initial_t0: initial t0 guess
        pos_lr, dir_lr, t0_lr, energy_lr: learning rates for different parameter types
        vertex_weight, wc_weight, energy_weight: scaling factors for loss components
    """
    
    # Stage 0: Energy estimation from photon count
    print("  Stage 0: Energy estimation from photon count")
    N_photons = jnp.sum(observed_charge)
    energy_guess = estimate_energy_from_photon_count(N_photons)
    print(f"    Observed photons: {N_photons}")
    print(f"    Energy guess: {energy_guess:.1f} (true: {true_energy:.1f})")
    
    # Stage 1: HCP lattice position grid search
    print("  Stage 1: HCP lattice position grid search")
    hcp_results = hcp_position_grid_search(
        hit_detector_positions, observed_times, observed_charge,
        true_position, TRUE_T0  # Use true t0 for initial position search
    )
    
    optimal_position = hcp_results['best_position']
    if optimal_position is None:
        print("  ERROR: HCP search failed to find valid position")
        return None
    
    # Stage 2: Hierarchical cone direction search
    print("  Stage 2: Hierarchical cone direction search at optimal position")
    cone_results = hierarchical_direction_search_cone(
        optimal_position, initial_t0, hit_detector_positions, observed_times, observed_charge,
        true_data, energy_guess
    )
    
    # Stage 3: Energy scan optimization
    print("  Stage 3: Energy scan optimization")
    energy_scan_results = energy_scan_optimization(
        optimal_position, cone_results['best_theta'], cone_results['best_phi'],
        initial_t0, hit_detector_positions, observed_times, observed_charge,
        true_data, energy_guess
    )
    
    optimal_energy = energy_scan_results['best_energy']
    
    # Stage 4: Combined gradient optimization
    print("  Stage 4: Combined gradient optimization (position + direction + t0 + energy)")
    
    # Create initial parameter vector with all optimized components
    initial_params = jnp.array([
        optimal_position[0], optimal_position[1], optimal_position[2],
        initial_t0,
        cone_results['best_theta'], cone_results['best_phi'],
        optimal_energy
    ])
    
    current_params = initial_params.copy()
    
    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [],
        'vertex_losses': [],
        'wc_losses': [],
        'energy_losses': [],
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': []
    }
    
    # Random key for WC loss evaluations
    opt_key = jax.random.PRNGKey(12345)
    
    print(f"    Starting gradient optimization from HCP+cone+energy search results...")
    
    for iteration in range(max_iterations):
        opt_key, _ = jax.random.split(opt_key)
        
        # Calculate loss and gradients
        combined_loss, grad = combined_grad_fn_with_energy(
            current_params, hit_detector_positions, observed_times, observed_charge,
            true_data, detector_params, opt_key, vertex_weight, wc_weight, energy_weight
        )
        
        # Get individual losses for tracking
        _, vertex_loss, wc_loss, energy_loss_val = combined_product_loss_with_energy(
            current_params, hit_detector_positions, observed_times, observed_charge,
            true_data, detector_params, opt_key, vertex_weight, wc_weight, energy_weight
        )
        
        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break
        
        # Extract gradients for different parameter types
        pos_grad = grad[:3]      # x, y, z gradients
        t0_grad = grad[3]        # t0 gradient
        theta_grad = grad[4]     # theta gradient
        phi_grad = grad[5]       # phi gradient
        energy_grad = grad[6]    # energy gradient
        
        # Apply different learning rates
        overall_lr = 1.0
        pos_step = -pos_lr * pos_grad * overall_lr
        t0_step = -t0_lr * t0_grad * overall_lr * 5.0
        theta_step = -dir_lr * theta_grad * overall_lr
        phi_step = -dir_lr * phi_grad * overall_lr
        energy_step = -energy_lr * energy_grad * overall_lr
        
        # Update parameters
        step = jnp.concatenate([pos_step, jnp.array([t0_step, theta_step, phi_step, energy_step])])
        current_params = current_params + step
        
        # Apply constraints
        current_params = jnp.array([
            jnp.clip(current_params[0], -DETECTOR_R * 0.9, DETECTOR_R * 0.9),  # x
            jnp.clip(current_params[1], -DETECTOR_R * 0.9, DETECTOR_R * 0.9),  # y
            jnp.clip(current_params[2], -DETECTOR_H/2 * 0.9, DETECTOR_H/2 * 0.9),  # z
            jnp.clip(current_params[3], -20.0, 20.0),  # t0
            current_params[4],  # theta
            current_params[5],  # phi
            jnp.clip(current_params[6], 100.0, 2000.0)  # energy (reasonable bounds)
        ])
        
        # Calculate current errors for tracking
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_theta = current_params[4]
        current_phi = current_params[5]
        current_energy = current_params[6]
        current_direction = spherical_to_cartesian(current_theta, current_phi)
        
        position_error = jnp.linalg.norm(current_position - true_position)
        t0_error = abs(current_t0 - TRUE_T0)
        energy_error = abs(current_energy - true_energy)
        cos_angle = jnp.clip(jnp.dot(current_direction, true_direction), -1.0, 1.0)
        direction_error = np.degrees(np.arccos(cos_angle))

        #print((iteration+1) % 100)
        # print((iteration+1) % 100 == 0)
        # if (iteration+1) % 100 == 0:
        #     print(iteration)
        #     print('AAAAH')
        if (iteration+1) % 100 == 0 or iteration == 0:
            print(f"      Iter {iteration}: pos_err={position_error:.3f}m, t0_err={t0_error:.3f}, dir_err={direction_error:.3f}°, E_err={energy_error:.1f}")
            print(f"      Energy gra: {grad[6]:.6f}")
        
        # Store history
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['vertex_losses'].append(float(vertex_loss))
        history['wc_losses'].append(float(wc_loss))
        history['energy_losses'].append(float(energy_loss_val))
        history['position_errors'].append(float(position_error))
        history['direction_errors'].append(float(direction_error))
        history['t0_errors'].append(float(t0_error))
        history['energy_errors'].append(float(energy_error))
    
    # Final calculations
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)
    
    final_position_error = jnp.linalg.norm(final_position - true_position)
    final_t0_error = abs(final_t0 - TRUE_T0)
    final_energy_error = abs(final_energy - true_energy)
    final_cos_angle = jnp.clip(jnp.dot(final_direction, true_direction), -1.0, 1.0)
    final_direction_error = np.degrees(np.arccos(final_cos_angle))
    
    return {
        'energy_estimation': {
            'n_photons': N_photons,
            'energy_guess': float(energy_guess),
            'energy_guess_error': float(abs(energy_guess - true_energy))
        },
        'hcp_position_search': hcp_results,
        'cone_direction_search': cone_results,
        'energy_scan_search': energy_scan_results,
        'initial_params': initial_params,
        'final_position': final_position,
        'final_direction': final_direction,
        'final_theta': final_theta,
        'final_phi': final_phi,
        'final_t0': final_t0,
        'final_energy': final_energy,
        'final_combined_loss': history['combined_losses'][-1] if history['combined_losses'] else float('inf'),
        'final_vertex_loss': history['vertex_losses'][-1] if history['vertex_losses'] else float('inf'),
        'final_wc_loss': history['wc_losses'][-1] if history['wc_losses'] else float('inf'),
        'final_energy_loss': history['energy_losses'][-1] if history['energy_losses'] else float('inf'),
        'final_position_error': float(final_position_error),
        'final_direction_error': float(final_direction_error),
        'final_t0_error': float(final_t0_error),
        'final_energy_error': float(final_energy_error),
        'total_iterations': len(history['parameters']) - 1,
        'converged': grad_norm < tolerance,
        'history': history
    }

print("Complete HCP position + cone direction + energy optimization function defined")

In [ ]:
def generate_event_data(event_idx, random_key):
    """
    Generate a single event with random parameters within detector bounds
    """
    entry_idx = event_idx % n_entries
    
    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)
    photon_data['N'] = len(photon_data['photon_origins'])
    
    key = random_key
    fraction = 0.6
    
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=DETECTOR_R * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-DETECTOR_H/2 * fraction, 
                               maxval=DETECTOR_H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])
    
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])
    
    true_energy = photon_data['energy']
    TRUE_T0 = jax.random.uniform(key, shape=(), minval=-3.0, maxval=3.0)
    
    true_params = (true_energy, true_position, true_direction)
    
    key, _ = jax.random.split(key)
    true_data = jax.lax.stop_gradient(data_simulator(true_params, detector_params, key, photon_data))
    
    hit_counts, hit_times_raw = true_data
    hit_times = hit_times_raw + TRUE_T0
    
    hit_mask = hit_counts > -999
    hit_detector_positions = detector_points[hit_mask]
    observed_times = hit_times[hit_mask]
    observed_charge = hit_counts[hit_mask]
    
    return {
        'event_idx': event_idx,
        'entry_idx': entry_idx,
        'true_energy': float(true_energy),
        'true_position': np.array(true_position),
        'true_direction': np.array(true_direction),
        'TRUE_T0': float(TRUE_T0),
        'true_data': true_data,
        'hit_detector_positions': hit_detector_positions,
        'observed_times': observed_times,
        'observed_charge': observed_charge,
        'n_hits': int(jnp.sum(hit_mask))
    }

print("Event generation function defined")

In [ ]:
# Multi-event optimization pipeline with HCP position + hierarchical cone direction + energy search
print(f"Starting HCP position + hierarchical cone direction + energy optimization for {N_EVENTS} events...")
print("=" * 80)

# Storage for all event results
all_event_results = []

# Performance tracking arrays
energy_guess_errors = []
hcp_position_errors = []
cone_direction_errors = []  # Track initial cone search direction error
energy_scan_improvements = []
final_position_errors = []
final_direction_errors = []
final_t0_errors = []
final_energy_errors = []
final_combined_losses = []
final_vertex_losses = []
final_wc_losses = []
final_energy_losses = []
convergence_rates = []

# Generate random keys for all events
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

# Process each event
for event_idx in tqdm(range(N_EVENTS), desc="Processing events"):
    try:
        print(f"\n--- Processing Event {event_idx} ---")
        
        # Generate event data
        event_data = generate_event_data(event_idx, event_keys[event_idx])
        
        # Extract event parameters
        true_position = event_data['true_position']
        true_direction = event_data['true_direction']
        true_energy = event_data['true_energy']
        TRUE_T0 = event_data['TRUE_T0']
        true_data = event_data['true_data']
        hit_detector_positions = event_data['hit_detector_positions']
        observed_times = event_data['observed_times']
        observed_charge = event_data['observed_charge']
        
        # Convert true direction to spherical coordinates
        true_theta, true_phi = cartesian_to_spherical(true_direction)
        
        # Generate random initial guess for t0
        np.random.seed(42 + event_idx)
        
        # Random t0 with noise
        t0_noise = np.random.normal(0, 10.0)
        initial_t0 = TRUE_T0 + t0_noise
        initial_t0 = jnp.clip(initial_t0, -15.0, 15.0)
        
        print(f"  True position: {true_position}")
        print(f"  True direction: {true_direction}")
        print(f"  True energy: {true_energy:.1f}")
        print(f"  Initial t0 error: {abs(initial_t0 - TRUE_T0):.3f}")
        
        # Run complete optimization pipeline with energy
        print("  Running HCP position + hierarchical cone direction + energy optimization...")
        results = run_complete_hcp_cone_energy_optimization(
            initial_t0=initial_t0,
            hit_detector_positions=hit_detector_positions,
            observed_times=observed_times,
            observed_charge=observed_charge,
            true_data=true_data,
            true_energy=true_energy,
            true_position=true_position,
            true_direction=true_direction,
            TRUE_T0=TRUE_T0,
            max_iterations=MAX_ITERATIONS
        )
        
        if results is None:
            print(f"  ERROR: Optimization failed for event {event_idx}")
            continue
        
        # Calculate improvements from each stage
        energy_guess_error = results['energy_estimation']['energy_guess_error']
        hcp_position_error = results['hcp_position_search']['position_error']
        
        # Calculate cone search direction error
        cone_direction = results['cone_direction_search']['best_direction']
        cone_cos_angle = np.clip(np.dot(cone_direction, true_direction), -1.0, 1.0)
        cone_direction_error = np.degrees(np.arccos(cone_cos_angle))
        
        energy_scan_improvement = results['energy_scan_search']['energy_improvement']
        
        # Store results for this event
        event_result = {
            'event_data': event_data,
            'initial_t0': initial_t0,
            'true_theta': float(true_theta),
            'true_phi': float(true_phi),
            'energy_guess_error': energy_guess_error,
            'hcp_position_error': hcp_position_error,
            'cone_direction_error': cone_direction_error,
            'energy_scan_improvement': energy_scan_improvement,
            'optimization_results': results
        }
        all_event_results.append(event_result)
        
        # Track performance metrics
        energy_guess_errors.append(energy_guess_error)
        hcp_position_errors.append(hcp_position_error)
        cone_direction_errors.append(cone_direction_error)
        energy_scan_improvements.append(energy_scan_improvement)
        final_position_errors.append(results['final_position_error'])
        final_direction_errors.append(results['final_direction_error'])
        final_t0_errors.append(results['final_t0_error'])
        final_energy_errors.append(results['final_energy_error'])
        final_combined_losses.append(results['final_combined_loss'])
        final_vertex_losses.append(results['final_vertex_loss'])
        final_wc_losses.append(results['final_wc_loss'])
        final_energy_losses.append(results['final_energy_loss'])
        convergence_rates.append(1.0 if results['converged'] else 0.0)
        
        print(f"  Energy guess error: {energy_guess_error:.1f}")
        print(f"  HCP position error: {hcp_position_error:.3f}m")
        print(f"  Cone direction error: {cone_direction_error:.1f}°")
        print(f"  Energy scan improvement: {energy_scan_improvement:.1f}")
        print(f"  Final errors - Position: {results['final_position_error']:.3f}m, "
              f"Direction: {results['final_direction_error']:.1f}°, t0: {results['final_t0_error']:.3f}, "
              f"Energy: {results['final_energy_error']:.1f}")
        print(f"  Final values - Position: {results['final_position']}, "
              f"Energy: {results['final_energy']:.1f} (true: {true_energy:.1f})")
        print(f"  Final losses - Combined: {results['final_combined_loss']:.6f}, "
              f"Vertex: {results['final_vertex_loss']:.6f}, WC: {results['final_wc_loss']:.6f}, "
              f"Energy: {results['final_energy_loss']:.6f}")
        print(f"  Converged: {results['converged']}, Iterations: {results['total_iterations']}")
            
    except Exception as e:
        print(f"Error processing event {event_idx}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\nCompleted processing {len(all_event_results)} events successfully")

In [ ]:
# Performance Analysis and Statistics
print("=" * 80)
print("HCP POSITION + HIERARCHICAL CONE DIRECTION + ENERGY OPTIMIZATION ANALYSIS")
print("=" * 80)

# Convert to numpy arrays
energy_guess_errors = np.array(energy_guess_errors)
hcp_position_errors = np.array(hcp_position_errors)
cone_direction_errors = np.array(cone_direction_errors)
energy_scan_improvements = np.array(energy_scan_improvements)
final_position_errors = np.array(final_position_errors)
final_direction_errors = np.array(final_direction_errors)
final_t0_errors = np.array(final_t0_errors)
final_energy_errors = np.array(final_energy_errors)
final_combined_losses = np.array(final_combined_losses)
final_vertex_losses = np.array(final_vertex_losses)
final_wc_losses = np.array(final_wc_losses)
final_energy_losses = np.array(final_energy_losses)
convergence_rates = np.array(convergence_rates)

# Calculate statistics
def percentile_68(data):
    return np.percentile(data, 68)

# Energy guess error statistics
energy_guess_error_68 = percentile_68(energy_guess_errors)
energy_guess_error_mean = np.mean(energy_guess_errors)
energy_guess_error_std = np.std(energy_guess_errors)

# HCP position error statistics
hcp_pos_error_68 = percentile_68(hcp_position_errors)
hcp_pos_error_mean = np.mean(hcp_position_errors)
hcp_pos_error_std = np.std(hcp_position_errors)

# Cone direction error statistics
cone_dir_error_68 = percentile_68(cone_direction_errors)
cone_dir_error_mean = np.mean(cone_direction_errors)
cone_dir_error_std = np.std(cone_direction_errors)

# Energy scan improvement statistics
energy_scan_improvement_mean = np.mean(energy_scan_improvements)
energy_scan_improvement_std = np.std(energy_scan_improvements)

# Final position error statistics
pos_error_68 = percentile_68(final_position_errors)
pos_error_mean = np.mean(final_position_errors)
pos_error_std = np.std(final_position_errors)

# Final direction error statistics
dir_error_68 = percentile_68(final_direction_errors)
dir_error_mean = np.mean(final_direction_errors)
dir_error_std = np.std(final_direction_errors)

# t0 error statistics
t0_error_68 = percentile_68(final_t0_errors)
t0_error_mean = np.mean(final_t0_errors)
t0_error_std = np.std(final_t0_errors)

# Energy error statistics
energy_error_68 = percentile_68(final_energy_errors)
energy_error_mean = np.mean(final_energy_errors)
energy_error_std = np.std(final_energy_errors)

# Loss statistics
combined_loss_mean = np.mean(final_combined_losses)
vertex_loss_mean = np.mean(final_vertex_losses)
wc_loss_mean = np.mean(final_wc_losses)
energy_loss_mean = np.mean(final_energy_losses)

# Convergence rate
convergence_rate_pct = np.mean(convergence_rates) * 100

print(f"\n🔢 ENERGY ESTIMATION:")
print(f"  Energy guess error - Mean: {energy_guess_error_mean:.1f} ± {energy_guess_error_std:.1f}")
print(f"  Energy guess error - 68%: {energy_guess_error_68:.1f}")

print(f"\n🔍 HCP POSITION GRID SEARCH:")
print(f"  HCP position error - Mean: {hcp_pos_error_mean:.3f} ± {hcp_pos_error_std:.3f} m")
print(f"  HCP position error - 68%: {hcp_pos_error_68:.3f} m")

print(f"\n🎯 HIERARCHICAL CONE DIRECTION SEARCH:")
print(f"  Cone direction error - Mean: {cone_dir_error_mean:.1f}° ± {cone_dir_error_std:.1f}°")
print(f"  Cone direction error - 68%: {cone_dir_error_68:.1f}°")

print(f"\n⚡ ENERGY SCAN OPTIMIZATION:")
print(f"  Energy scan improvement - Mean: {energy_scan_improvement_mean:.1f} ± {energy_scan_improvement_std:.1f}")

print(f"\n🎯 FINAL POSITION RECONSTRUCTION:")
print(f"  Position error - Mean: {pos_error_mean:.3f} ± {pos_error_std:.3f} m")
print(f"  Position error - 68%: {pos_error_68:.3f} m")

print(f"\n🧭 FINAL DIRECTION RECONSTRUCTION:")
print(f"  Direction error - Mean: {dir_error_mean:.1f}° ± {dir_error_std:.1f}°")
print(f"  Direction error - 68%: {dir_error_68:.1f}°")

print(f"\n⏰ TIME RECONSTRUCTION:")
print(f"  t0 error - Mean: {t0_error_mean:.3f} ± {t0_error_std:.3f}")
print(f"  t0 error - 68%: {t0_error_68:.3f}")

print(f"\n🔢 ENERGY RECONSTRUCTION:")
print(f"  Energy error - Mean: {energy_error_mean:.1f} ± {energy_error_std:.1f}")
print(f"  Energy error - 68%: {energy_error_68:.1f}")

print(f"\n📊 LOSS ANALYSIS:")
print(f"  Combined loss - Mean: {combined_loss_mean:.6f}")
print(f"  Vertex loss - Mean: {vertex_loss_mean:.6f}")
print(f"  WC loss - Mean: {wc_loss_mean:.6f}")
print(f"  Energy loss - Mean: {energy_loss_mean:.6f}")

print(f"\n⚡ OPTIMIZATION PERFORMANCE:")
print(f"  Convergence rate: {convergence_rate_pct:.1f}%")

print(f"\n✨ KEY PERFORMANCE METRICS:")
print(f"  🔢 68% Energy Guess Error: {energy_guess_error_68:.1f}")
print(f"  🔍 68% HCP Position Error: {hcp_pos_error_68:.3f} m")
print(f"  🎯 68% Cone Direction Error: {cone_dir_error_68:.1f}°")
print(f"  🎯 68% Final Position Error: {pos_error_68:.3f} m")
print(f"  🧭 68% Final Direction Error: {dir_error_68:.1f}°")
print(f"  ⏰ 68% t0 Error: {t0_error_68:.3f}")
print(f"  🔢 68% Final Energy Error: {energy_error_68:.1f}")
print(f"  ⚡ Convergence Rate: {convergence_rate_pct:.1f}%")

# Calculate improvements
position_improvement = hcp_pos_error_mean - pos_error_mean
direction_improvement = cone_dir_error_mean - dir_error_mean
energy_improvement = energy_guess_error_mean - energy_error_mean

print(f"\n📈 OPTIMIZATION IMPROVEMENTS:")
print(f"  Average position improvement: {position_improvement:.3f}m (HCP → final)")
print(f"  Average direction improvement: {direction_improvement:.1f}° (cone → final)")
print(f"  Average energy improvement: {energy_improvement:.1f} (guess → final)")
print(f"  Position improvement ratio: {position_improvement/hcp_pos_error_mean*100:.1f}%")
print(f"  Direction improvement ratio: {direction_improvement/cone_dir_error_mean*100:.1f}%")
print(f"  Energy improvement ratio: {energy_improvement/energy_guess_error_mean*100:.1f}%")

print(f"\n🔬 Key innovations: HCP lattice position + hierarchical cone direction + energy optimization!")
print(f"    Energy estimation from photon count followed by energy scan and gradient optimization.")

In [ ]:
# Save detailed results
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

results_summary = {
    'config': {
        'n_events': N_EVENTS,
        'temperature': TEMPERATURE,
        'vertex_weight_scale': VERTEX_WEIGHT_SCALE,
        'wc_weight_scale': WC_WEIGHT_SCALE,
        'energy_weight_scale': ENERGY_WEIGHT_SCALE,
        'position_learning_rate': POSITION_LEARNING_RATE,
        'direction_learning_rate': DIRECTION_LEARNING_RATE,
        't0_learning_rate': T0_LEARNING_RATE,
        'energy_learning_rate': ENERGY_LEARNING_RATE,
        'hcp_l0': HCP_L0,
        'hcp_levels': HCP_LEVELS,
        'hcp_reduction': HCP_REDUCTION,
        'cone_levels': CONE_LEVELS,
        'cone_initial_div': CONE_INITIAL_DIV,
        'cone_max_angle_deg': CONE_MAX_ANGLE_DEG,
        'cone_reduction': CONE_REDUCTION,
        'energy_delta': ENERGY_DELTA,
        'energy_scan_steps': ENERGY_SCAN_STEPS,
        'detector_r': float(DETECTOR_R),
        'detector_h': float(DETECTOR_H),
        'data_file': data_file,
        'detector_file': default_json_filename
    },
    'performance_metrics': {
        'energy_guess_error_68pct': float(energy_guess_error_68),
        'energy_guess_error_mean': float(energy_guess_error_mean),
        'energy_guess_error_std': float(energy_guess_error_std),
        'hcp_position_error_68pct': float(hcp_pos_error_68),
        'hcp_position_error_mean': float(hcp_pos_error_mean),
        'hcp_position_error_std': float(hcp_pos_error_std),
        'cone_direction_error_68pct': float(cone_dir_error_68),
        'cone_direction_error_mean': float(cone_dir_error_mean),
        'cone_direction_error_std': float(cone_dir_error_std),
        'energy_scan_improvement_mean': float(energy_scan_improvement_mean),
        'energy_scan_improvement_std': float(energy_scan_improvement_std),
        'position_error_68pct': float(pos_error_68),
        'position_error_mean': float(pos_error_mean),
        'position_error_std': float(pos_error_std),
        'direction_error_68pct': float(dir_error_68),
        'direction_error_mean': float(dir_error_mean),
        'direction_error_std': float(dir_error_std),
        't0_error_68pct': float(t0_error_68),
        't0_error_mean': float(t0_error_mean),
        't0_error_std': float(t0_error_std),
        'energy_error_68pct': float(energy_error_68),
        'energy_error_mean': float(energy_error_mean),
        'energy_error_std': float(energy_error_std),
        'combined_loss_mean': float(combined_loss_mean),
        'vertex_loss_mean': float(vertex_loss_mean),
        'wc_loss_mean': float(wc_loss_mean),
        'energy_loss_mean': float(energy_loss_mean),
        'convergence_rate': float(convergence_rate_pct) / 100,
        'position_improvement_mean': float(position_improvement),
        'direction_improvement_mean': float(direction_improvement),
        'energy_improvement_mean': float(energy_improvement)
    },
    'raw_data': {
        'energy_guess_errors': energy_guess_errors.tolist(),
        'hcp_position_errors': hcp_position_errors.tolist(),
        'cone_direction_errors': cone_direction_errors.tolist(),
        'energy_scan_improvements': energy_scan_improvements.tolist(),
        'final_position_errors': final_position_errors.tolist(),
        'final_direction_errors': final_direction_errors.tolist(),
        'final_t0_errors': final_t0_errors.tolist(),
        'final_energy_errors': final_energy_errors.tolist(),
        'final_combined_losses': final_combined_losses.tolist(),
        'final_vertex_losses': final_vertex_losses.tolist(),
        'final_wc_losses': final_wc_losses.tolist(),
        'final_energy_losses': final_energy_losses.tolist(),
        'convergence_rates': convergence_rates.tolist()
    },
    'all_event_results': all_event_results  # Complete detailed results
}

# Save results
output_file = output_dir / f'hcp_cone_energy_optimization_{N_EVENTS}_results.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(results_summary, f)

print(f"\n💾 Results saved:")
print(f"  - Complete results: {output_file}")

print(f"\n🎉 HCP POSITION + HIERARCHICAL CONE DIRECTION + ENERGY OPTIMIZATION ANALYSIS COMPLETE!")
print(f"\n🎯 KEY FINDINGS:")
print(f"  • Energy guess 68% error: {energy_guess_error_68:.1f}")
print(f"  • HCP 68% position error: {hcp_pos_error_68:.3f} m")
print(f"  • Cone 68% direction error: {cone_dir_error_68:.1f}°")
print(f"  • Final 68% position error: {pos_error_68:.3f} m")
print(f"  • Final 68% direction error: {dir_error_68:.1f}°")
print(f"  • Final 68% t0 error: {t0_error_68:.3f}")
print(f"  • Final 68% energy error: {energy_error_68:.1f}")
print(f"  • Convergence rate: {convergence_rate_pct:.1f}%")
print(f"  • Mean combined loss: {combined_loss_mean:.6f}")
print(f"  • Average position improvement: {position_improvement:.3f}m")
print(f"  • Average direction improvement: {direction_improvement:.1f}°")
print(f"  • Average energy improvement: {energy_improvement:.1f}")

print(f"\n📈 Energy optimization provides superior energy reconstruction from photon count estimates!")
print(f"\n🔬 Key innovations: HCP lattice + hierarchical cone search + energy scan + combined optimization")
print(f"    Four-stage optimization: position → direction → energy → combined gradient optimization.")